# Predicting Moral Values in Text
### This Code offers predicting moral values from the MoralBERT weights deployad in Hugging Face.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    PretrainedConfig,
    PreTrainedModel
)
from sklearn.utils import resample
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    multilabel_confusion_matrix as mcm,
    precision_score,
    recall_score
)
from transformers.modeling_outputs import SequenceClassifierOutput

In [3]:
import os
import random
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from itertools import product
from scipy.special import softmax
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    multilabel_confusion_matrix as mcm,
    precision_score,
    recall_score
)
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from tabulate import tabulate
from torch.autograd import Function
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler, TensorDataset
from tqdm import trange
from tqdm.auto import trange
from transformers import (
    AutoConfig,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BertModel,
    PretrainedConfig,
    PreTrainedModel,
    get_linear_schedule_with_warmup
)
from transformers.modeling_outputs import SequenceClassifierOutput

In [4]:
base_model = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(base_model)

HF_EXPORT_ROOT = Path("/content/drive/MyDrive/moralbert_inggris_v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
from transformers import PretrainedConfig, PreTrainedModel, AutoModel, AutoConfig
from transformers.modeling_outputs import SequenceClassifierOutput
import torch
import torch.nn as nn

class PlainBERTConfig(PretrainedConfig):
    model_type = "plainbert"

    def __init__(
        self,
        base_model_name_or_path="bert-base-uncased",
        num_labels=2,
        class_weight=None,
        identity_weight=0.0,
        reconstruction_weight=0.0,
        moral_weight=1.0,
        freeze_bert=False,
        id2label=None,
        label2id=None,
        **kwargs,
    ):
        super().__init__(num_labels=num_labels, id2label=id2label, label2id=label2id, **kwargs)
        self.base_model_name_or_path = base_model_name_or_path
        self.num_labels = num_labels
        self.class_weight = class_weight if class_weight is not None else [1.0] * num_labels
        self.identity_weight = float(identity_weight)
        self.reconstruction_weight = float(reconstruction_weight)
        self.moral_weight = float(moral_weight)
        self.freeze_bert = bool(freeze_bert)
        self.problem_type = "single_label_classification"


class PlainBERTForSequenceClassification(PreTrainedModel):
    config_class = PlainBERTConfig
    base_model_prefix = "bert"
    supports_gradient_checkpointing = False

    def __init__(self, config):
        super().__init__(config)

        self.num_labels = config.num_labels
        self.freeze = config.freeze_bert

        base_encoder_config = AutoConfig.from_pretrained(config.base_model_name_or_path)
        self.bert = AutoModel.from_config(base_encoder_config)
        bert_dim = self.bert.config.hidden_size

        self.invariant_trans = nn.Linear(bert_dim, bert_dim)

        if config.identity_weight + config.reconstruction_weight == 0:
            self.moral_classification = nn.Linear(bert_dim, config.num_labels)
        else:
            self.moral_classification = nn.Sequential(
                nn.Linear(bert_dim, bert_dim),
                nn.ReLU(),
                nn.Linear(bert_dim, config.num_labels),
            )

        class_weight = config.class_weight if config.class_weight is not None else [1.0] * config.num_labels
        if class_weight and class_weight[0] > 0:
            weights = torch.tensor(class_weight).float()
        else:
            weights = torch.ones(config.num_labels).float()
        self.register_buffer("class_weights", weights)

        self.reconstruction_feed = nn.Linear(bert_dim, bert_dim)
        self.loss_reconstruction = nn.MSELoss()
        self.register_buffer("identity", torch.eye(bert_dim))

        try:
            self.post_init()
        except Exception as e:
            print("Warning during post_init:", e)

    def forward(
        self,
        input_ids=None,
        token_type_ids=None,
        attention_mask=None,
        labels=None,
        original_bert_embeddings=None,
        **kwargs,
    ):
        if self.freeze:
            with torch.no_grad():
                cls = self.bert(
                    input_ids=input_ids,
                    token_type_ids=token_type_ids,
                    attention_mask=attention_mask
                ).last_hidden_state[:, 0, :]
        else:
            cls = self.bert(
                input_ids=input_ids,
                token_type_ids=token_type_ids,
                attention_mask=attention_mask
            ).last_hidden_state[:, 0, :]

        z = self.invariant_trans(cls)
        logits = self.moral_classification(z)

        total_loss = None
        if labels is not None:
            loss_fn_moral = nn.CrossEntropyLoss(weight=self.class_weights)
            loss_moral = loss_fn_moral(logits, labels)

            if original_bert_embeddings is not None and self.config.reconstruction_weight > 0:
                loss_recon = self.loss_reconstruction(
                    self.reconstruction_feed(z),
                    original_bert_embeddings
                ) * self.config.reconstruction_weight
            else:
                loss_recon = 0.0

            if self.config.identity_weight > 0:
                loss_identity = torch.norm(
                    self.invariant_trans.weight - self.identity
                ) * self.config.identity_weight
            else:
                loss_identity = 0.0

            total_loss = (loss_moral * self.config.moral_weight) + loss_recon + loss_identity

        return SequenceClassifierOutput(loss=total_loss, logits=logits)

In [6]:
def preprocessing(input_text, tokenizer):
    return tokenizer(
        input_text,
        add_special_tokens=True,
        max_length=150,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors="pt"
    )

In [7]:
possible_labels = [
    "care", "harm",
    "fairness", "cheating",
    "loyalty", "betrayal",
    "authority", "subversion",
    "purity", "degradation",
    "liberty", "oppression"
]

In [8]:
for lab in possible_labels:
    checkpoint_folder = HF_EXPORT_ROOT / lab.replace(" ", "_")
    print(lab, "->", checkpoint_folder)
    print("exists:", checkpoint_folder.exists(), "| is_dir:", checkpoint_folder.is_dir())

care -> /content/drive/MyDrive/moralbert_inggris_v2/care
exists: True | is_dir: True
harm -> /content/drive/MyDrive/moralbert_inggris_v2/harm
exists: True | is_dir: True
fairness -> /content/drive/MyDrive/moralbert_inggris_v2/fairness
exists: True | is_dir: True
cheating -> /content/drive/MyDrive/moralbert_inggris_v2/cheating
exists: True | is_dir: True
loyalty -> /content/drive/MyDrive/moralbert_inggris_v2/loyalty
exists: True | is_dir: True
betrayal -> /content/drive/MyDrive/moralbert_inggris_v2/betrayal
exists: True | is_dir: True
authority -> /content/drive/MyDrive/moralbert_inggris_v2/authority
exists: True | is_dir: True
subversion -> /content/drive/MyDrive/moralbert_inggris_v2/subversion
exists: True | is_dir: True
purity -> /content/drive/MyDrive/moralbert_inggris_v2/purity
exists: True | is_dir: True
degradation -> /content/drive/MyDrive/moralbert_inggris_v2/degradation
exists: True | is_dir: True
liberty -> /content/drive/MyDrive/moralbert_inggris_v2/liberty
exists: True | is

In [9]:
import os
import __main__

dummy_py_path = os.path.join(os.getcwd(), "notebook_session.py")
if not os.path.exists(dummy_py_path):
    with open(dummy_py_path, "w", encoding="utf-8") as f:
        f.write("# dummy file for transformers custom model in notebook\n")

__main__.__file__ = dummy_py_path

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loaded_models = {}

for lab in possible_labels:
    checkpoint_folder = HF_EXPORT_ROOT / lab.replace(" ", "_")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = PlainBERTForSequenceClassification.from_pretrained(
        str(checkpoint_folder),
        local_files_only=True
    ).to(device)

    model.eval()
    loaded_models[lab] = model

print("All models loaded successfully.")
print(checkpoint_folder)
print(os.listdir(checkpoint_folder))

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

All models loaded successfully.
/content/drive/MyDrive/moralbert_inggris_v2/oppression
['config.json', 'model.safetensors', 'README.md', 'tokenizer.json', 'configuration_plainbert.py', 'tokenizer_config.json', 'modeling_plainbert.py']


In [11]:
def get_model_score(sentence, label_name):
    model = loaded_models[label_name]

    encodeds = preprocessing(sentence, tokenizer)
    encodeds = {k: v.to(device) for k, v in encodeds.items()}

    with torch.no_grad():
        outputs = model(**encodeds)
        probs = F.softmax(outputs.logits, dim=1)

    score_positive = probs[0, 1].item()
    score_negative = probs[0, 0].item()

    return {
        "score_positive": score_positive,
        "score_negative": score_negative,
        "pred_label": int(score_positive >= 0.5)
    }

In [12]:
test_df = pd.read_csv("test_inggris_v2.csv")
sentences = test_df["sentence"].tolist()

In [13]:
results = []

for sentence in sentences:
    row = {"sentence": sentence}

    for lab in possible_labels:
        output = get_model_score(sentence, lab)
        row[f"{lab}_score"] = output["score_positive"]
        #row[f"{lab}_pred"] = output["pred_label"]

    results.append(row)

results_df = pd.DataFrame(results)
results_df

,sentence,care_score,harm_score,fairness_score,cheating_score,loyalty_score,betrayal_score,authority_score,subversion_score,purity_score,degradation_score,liberty_score,oppression_score
0,After a prolonged Experience he came to know t...,0.000214,0.000219,0.000294,0.000215,0.000245,0.000368,0.000437,0.010098,0.000150,0.000288,0.000764,0.000385
1,Jolly Story of the Slums.,0.000201,0.000200,0.000336,0.000170,0.000225,0.000325,0.000297,0.008284,0.000143,0.000275,0.000774,0.000384
2,The Book that begins with a twenty-page Descri...,0.000227,0.999752,0.000330,0.000204,0.000279,0.000378,0.000642,0.009564,0.000185,0.000352,0.000815,0.000609
3,It condenses the whole Plot and dishes up the ...,0.000190,0.000212,0.000318,0.000181,0.000230,0.000332,0.000304,0.008527,0.000146,0.000297,0.000764,0.000407
4,"After that, who would have the Nerve to wade t...",0.000232,0.000219,0.000385,0.000181,0.000237,0.000318,0.000334,0.008592,0.000153,0.000269,0.000776,0.000392
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4632,Who the inhabitantsWho with a slender pittance...,0.000240,0.000203,0.000291,0.000184,0.000245,0.000438,0.000327,0.010322,0.000142,0.000296,0.000750,0.000469
4633,"There his other daughter, Ismene, joined him, ...",0.000217,0.000218,0.000286,0.000210,0.000262,0.000314,0.999382,0.010228,0.000151,0.000284,0.000799,0.000364
4634,Polynices had been expelled from Thebes by his...,0.998544,0.000264,0.999318,0.000302,0.000292,0.043057,0.001769,0.018325,0.000169,0.000370,0.002415,0.017741
4635,"She was seized by the guard, and led before Cr...",0.000210,0.000220,0.000285,0.000289,0.000249,0.000325,0.000368,0.012695,0.000133,0.000278,0.000775,0.000519


In [14]:
input_files = test_df["sentence"].values

tokenizer = AutoTokenizer.from_pretrained(base_model)

original_input_id = []
original_attention_masks = []
original_token_type_id = []

def preprocessing(input_text, tokenizer):
    return tokenizer(
        input_text,
        add_special_tokens=True,
        max_length=150,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_tensors="pt"
    )

for sample in input_files:
    original_encoding_dict = preprocessing(sample, tokenizer)

    original_input_id.append(original_encoding_dict["input_ids"])
    original_attention_masks.append(original_encoding_dict["attention_mask"])

    if "token_type_ids" in original_encoding_dict:
        original_token_type_id.append(original_encoding_dict["token_type_ids"])
    else:
        original_token_type_id.append(torch.zeros_like(original_encoding_dict["input_ids"]))

original_input_id = torch.cat(original_input_id, dim=0)
original_token_type_id = torch.cat(original_token_type_id, dim=0)
original_attention_masks = torch.cat(original_attention_masks, dim=0)

print("recreated")
print("original_input_id.is_meta =", original_input_id.is_meta)
print("original_token_type_id.is_meta =", original_token_type_id.is_meta)
print("original_attention_masks.is_meta =", original_attention_masks.is_meta)

recreated
original_input_id.is_meta = False
original_token_type_id.is_meta = False
original_attention_masks.is_meta = False


In [15]:
possible_labels = [
    "care", "harm",
    "fairness", "cheating",
    "loyalty", "betrayal",
    "authority", "subversion",
    "purity", "degradation",
    "liberty", "oppression"
]

batch_size = 16

torch.set_default_device("cpu")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

predictions = []
threshold_summary = []

for lab_idx, lab in enumerate(possible_labels):
    print(f"\nProcessing label: {lab}")

    best_f1 = 0
    best_th = 0.5
    best_y = None

    true_labels_for_lab = test_df[lab].astype(int).tolist()

    val_set = TensorDataset(
        original_input_id,
        original_token_type_id,
        original_attention_masks,
        torch.tensor(true_labels_for_lab, dtype=torch.long)
    )

    validation_dataloader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

    checkpoint_folder = HF_EXPORT_ROOT / lab.replace(" ", "_")

    model = PlainBERTForSequenceClassification.from_pretrained(
        str(checkpoint_folder),
        local_files_only=True
    )
    model = model.to(device)
    model.eval()

    all_probs = []
    y_true = []

    for batch in validation_dataloader:
        b_input_ids, b_token_type_ids, b_input_mask, b_labels = [
            t.to(device) for t in batch
        ]

        model_inputs = {
            "input_ids": b_input_ids,
            "attention_mask": b_input_mask
        }

        if b_token_type_ids is not None:
            model_inputs["token_type_ids"] = b_token_type_ids

        with torch.no_grad():
            outputs = model(**model_inputs)

            logits = outputs.logits.detach().cpu()
            probs = torch.softmax(logits, dim=1).numpy()[:, 1]

            all_probs.extend(probs.tolist())
            y_true.extend(b_labels.detach().cpu().numpy().tolist())

    all_probs = np.array(all_probs)
    y_true = np.array(y_true)

    for th in np.arange(0.05, 1.00, 0.05):
        y_pred = (all_probs >= th).astype(int)
        f1 = f1_score(y_true, y_pred, average="binary", zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_th = th
            best_y = y_pred.copy()

    threshold_summary.append({
        "label": lab,
        "best_threshold": float(best_th),
        "best_f1": float(best_f1)
    })

    if lab_idx == 0:
        for ex_id, (pred_label, true_label) in enumerate(zip(best_y, y_true)):
            predictions.append({
                "id": ex_id,
                f"pred_{lab}": int(pred_label),
                f"true_{lab}": int(true_label)
            })
    else:
        for ex_id, (pred_label, true_label) in enumerate(zip(best_y, y_true)):
            predictions[ex_id][f"pred_{lab}"] = int(pred_label)
            predictions[ex_id][f"true_{lab}"] = int(true_label)

    print("Evaluation")
    print(f"{lab} | best threshold: {best_th:.2f} | best F1: {best_f1:.4f}")

    target_names = [f"Non-{lab}", lab]
    report = classification_report(
        y_true,
        best_y,
        target_names=target_names,
        zero_division=0
    )

    print("\nClassification Report:")
    print(report)

pred_df = pd.DataFrame(predictions)
threshold_df = pd.DataFrame(threshold_summary)


Processing label: care


Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Evaluation
care | best threshold: 0.90 | best F1: 0.9896

Classification Report:
              precision    recall  f1-score   support

    Non-care       1.00      1.00      1.00      4494
        care       0.98      1.00      0.99       143

    accuracy                           1.00      4637
   macro avg       0.99      1.00      0.99      4637
weighted avg       1.00      1.00      1.00      4637


Processing label: harm


Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Evaluation
harm | best threshold: 0.95 | best F1: 0.9964

Classification Report:
              precision    recall  f1-score   support

    Non-harm       1.00      1.00      1.00      4360
        harm       0.99      1.00      1.00       277

    accuracy                           1.00      4637
   macro avg       1.00      1.00      1.00      4637
weighted avg       1.00      1.00      1.00      4637


Processing label: fairness


...

Evaluation
betrayal | best threshold: 0.95 | best F1: 0.9412

Classification Report:
              precision    recall  f1-score   support

Non-betrayal       1.00      1.00      1.00      4619
    betrayal       1.00      0.89      0.94        18

    accuracy                           1.00      4637
   macro avg       1.00      0.94      0.97      4637
weighted avg       1.00      1.00      1.00      4637


Processing label: authority


Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Evaluation
authority | best threshold: 0.35 | best F1: 0.9835

Classification Report:
               precision    recall  f1-score   support

Non-authority       1.00      1.00      1.00      4486
    authority       0.98      0.99      0.98       151

     accuracy                           1.00      4637
    macro avg       0.99      0.99      0.99      4637
 weighted avg       1.00      1.00      1.00      4637


Processing label: subversion


Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

TypeError: 'NoneType' object is not iterable

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
results = []

for idx_lab, lab in enumerate(possible_labels):
    result = {"Moral Value": lab}
    true = test_df[lab].values
    candidate = pred_df["pred_" + lab].values

    result["F1 Score (Binary)"] = f1_score(true, candidate, average="binary", zero_division=0)
    result["F1 Score (Weighted)"] = f1_score(true, candidate, average="weighted", zero_division=0)

    result["Precision Score (Binary)"] = precision_score(true, candidate, average="binary", zero_division=0)
    result["Precision Score (Weighted)"] = precision_score(true, candidate, average="weighted", zero_division=0)

    result["Recall Score (Binary)"] = recall_score(true, candidate, average="binary", zero_division=0)
    result["Recall Score (Weighted)"] = recall_score(true, candidate, average="weighted", zero_division=0)

    result["Accuracy"] = accuracy_score(true, candidate)

    results.append(result)

results = pd.DataFrame(results)

In [ ]:
possible_labels = [
    "care", "harm",
    "fairness", "cheating",
    "loyalty", "betrayal",
    "authority", "subversion",
    "purity", "degradation",
    "liberty", "oppression"
]

test_df.reset_index(drop=True, inplace=True)
n_bootstrap_iters = 1000

bootstrap_results = {
    label: {
        metric: [] for metric in [
            "F1 (Binary)", "F1 (Macro)", "F1 (Weighted)",
            "Precision (Binary)", "Precision (Macro)", "Precision (Weighted)",
            "Recall (Binary)", "Recall (Macro)", "Recall (Weighted)",
            "Accuracy"
        ]
    } for label in possible_labels
}

for _ in range(n_bootstrap_iters):
    for lab in possible_labels:
        sample_indices = resample(np.arange(len(test_df)), replace=True)
        true = test_df.loc[sample_indices, lab].values
        candidate = pred_df.loc[sample_indices, f"pred_{lab}"].values

        bootstrap_results[lab]["F1 (Binary)"].append(f1_score(true, candidate, average="binary", zero_division=0))
        bootstrap_results[lab]["F1 (Macro)"].append(f1_score(true, candidate, average="macro", zero_division=0))
        bootstrap_results[lab]["F1 (Weighted)"].append(f1_score(true, candidate, average="weighted", zero_division=0))
        bootstrap_results[lab]["Precision (Binary)"].append(precision_score(true, candidate, average="binary", zero_division=0))
        bootstrap_results[lab]["Precision (Macro)"].append(precision_score(true, candidate, average="macro", zero_division=0))
        bootstrap_results[lab]["Precision (Weighted)"].append(precision_score(true, candidate, average="weighted", zero_division=0))
        bootstrap_results[lab]["Recall (Binary)"].append(recall_score(true, candidate, average="binary", zero_division=0))
        bootstrap_results[lab]["Recall (Macro)"].append(recall_score(true, candidate, average="macro", zero_division=0))
        bootstrap_results[lab]["Recall (Weighted)"].append(recall_score(true, candidate, average="weighted", zero_division=0))
        bootstrap_results[lab]["Accuracy"].append(accuracy_score(true, candidate))

std_devs = {
    label: {metric: np.std(values) for metric, values in metrics.items()}
    for label, metrics in bootstrap_results.items()
}

final_results = []
for lab in possible_labels:
    result = {"Moral Value": lab}
    true = test_df[lab].values
    candidate = pred_df[f"pred_{lab}"].values

    result["F1 Score (Binary)"] = f"{f1_score(true, candidate, average='binary', zero_division=0):.2f} ± {std_devs[lab]['F1 (Binary)']:.2f}"
    result["F1 Score (Macro)"] = f"{f1_score(true, candidate, average='macro', zero_division=0):.2f} ± {std_devs[lab]['F1 (Macro)']:.2f}"
    result["F1 Score (Weighted)"] = f"{f1_score(true, candidate, average='weighted', zero_division=0):.2f} ± {std_devs[lab]['F1 (Weighted)']:.2f}"

    result["Precision Score (Binary)"] = f"{precision_score(true, candidate, average='binary', zero_division=0):.2f} ± {std_devs[lab]['Precision (Binary)']:.2f}"
    result["Precision Score (Macro)"] = f"{precision_score(true, candidate, average='macro', zero_division=0):.2f} ± {std_devs[lab]['Precision (Macro)']:.2f}"
    result["Precision Score (Weighted)"] = f"{precision_score(true, candidate, average='weighted', zero_division=0):.2f} ± {std_devs[lab]['Precision (Weighted)']:.2f}"

    result["Recall Score (Binary)"] = f"{recall_score(true, candidate, average='binary', zero_division=0):.2f} ± {std_devs[lab]['Recall (Binary)']:.2f}"
    result["Recall Score (Macro)"] = f"{recall_score(true, candidate, average='macro', zero_division=0):.2f} ± {std_devs[lab]['Recall (Macro)']:.2f}"
    result["Recall Score (Weighted)"] = f"{recall_score(true, candidate, average='weighted', zero_division=0):.2f} ± {std_devs[lab]['Recall (Weighted)']:.2f}"

    result["Accuracy"] = f"{accuracy_score(true, candidate):.2f} ± {std_devs[lab]['Accuracy']:.2f}"

    final_results.append(result)

results_df = pd.DataFrame(final_results)
results_df